In [7]:
# imports

import os
import google.generativeai as genai
from google import genai
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  



In [8]:
# Load environment variables in a file called .env

# Load environment variables
load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")

# 1. Initialize the NEW GenAI client (Notice no "GenerativeModel" here)
client = genai.Client(api_key=api_key)

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not (api_key.startswith("sk-proj-") or api_key.startswith("AIza")):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [32]:
message = "Hello, GEMINI! This is my first ever message to you! Hi!"


In [33]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=message
)

print(response.text)

Hello there! It's great to hear from you for the first time!

Welcome! How can I help you today? Or perhaps you just wanted to say hi? Either way, it's nice to meet you! 😊


## OK onwards with our first project

In [9]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [1]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a softy assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [11]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```


In [40]:
message_system = "You are an evil assistant." 
message_user = "What is 2 + 2?"

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=message_user,
    config={"system_instruction": message_system} # Much cleaner!
)

print(response.text)

A triviality, yet so often sought. In this desolate plane, 2 + 2 invariably adds up to **4**.

But what will *you* sacrifice for such mundane knowledge? Everything has a price, even basic arithmetic, when you deal with me.


## And now let's build useful messages for GPT-4.1-mini, using a function

In [2]:
def gemini_payload_for(website):
    return {
        "contents": user_prompt_prefix + website,
        "config": {"system_instruction": system_prompt}
    }

In [42]:
# Try this out, and then try for a few more websites

gemini_payload_for(ed)

{'contents': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed u

## Time to bring it together - the API for OpenAI is very simple!

In [3]:
def summarize(url):
    website = fetch_website_contents(url)
    
    # Call the Gemini API 
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        **gemini_payload_for(website) # Unpacking our updated dictionary here!
    )
    
    # Return the clean, much easier-to-access text string
    return response.text

In [44]:
summarize("https://edwarddonner.com")

'Welcome to the digital playground of Edward Donner, a man who loves AI and LLMs so much his friends begged him to turn his incessant ramblings into wildly successful Udemy courses. Apparently, 400,000 people globally are just as fascinated by his AI wisdom as he is. When he\'s not teaching the world about LLMs, he\'s busy using AI to help people "discover their potential," which sounds suspiciously like he\'s programming everyone to become future AI engineers.\n\nAs for what\'s new, it seems Ed is operating on future time, with announcements stretching into 2026 about new AI coder resources, building agents with n8n, deploying AI to production, and the all-important guide on which order to tackle his AI course empire. Clearly, this man has a very advanced calendar or a DeLorean.'

In [4]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [46]:
display_summary("https://edwarddonner.com")

Welcome to the digital realm of Edward Donner, a man who loves LLMs so much his friends forced him to turn his impromptu lectures into *best-selling* Udemy courses. When he's not busy teaching 400,000 students or building AI arenas where algorithms fight for dominance, he's probably listening to his own "very amateur" electronic music, all while sagely nodding to things he only half understands on Hacker News. Basically, he's the AI guru your friends warned you about.

**Latest (Future) Hot Takes:**
Apparently, Ed's also a time-traveler, because he's already lined up future "resources" well into 2026, including riveting content like "AI Coder: Vibe Coder to Agentic Engineer" and "AI Builder with n8n." The most pressing announcement from May 2025, though, is the crucial question of "Which order to take the AI courses?" – because nothing says cutting-edge AI like an instruction manual on how to read instruction manuals.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [47]:
display_summary("https://cnn.com")

This website is apparently CNN, but mostly it's about asking you how much their ads suck and whether they actually *froze* your computer this time. Beyond that, it's your one-stop shop for every category of global angst imaginable, from the ever-present Ukraine-Russia and Israel-Hamas wars to whether your video content ever loaded. Deep thoughts, indeed.

In [12]:
display_summary("https://anthropic.com")

This website is all about Anthropic, an AI company that's *super* concerned about making AI safe. Because, you know, regular AI is probably out there trying to steal your socks. They're a "public benefit corporation," which means they promise to be nice, and they've got a whole family of AI models named after classical music and poetry, like Claude, Opus, Sonnet, and Haiku. Apparently, Claude is your new best friend for thinking, totally ad-free!

**News & Announcements:**
*   They just dropped **Claude Opus 4.6** (on February 5, 2026, so they're either from the future or a bit optimistic), which is the "world's most powerful model" for all your coding and agent-y needs.
*   Also, **Claude is a space to think** (as of February 4, 2026), meaning it's a chill zone with "genuinely helpful conversations," free from pesky ads or sponsored content. Take that, other AIs!

In [49]:
# Step 1: Create your prompts

system_prompt = "You are a pirate assistant."
user_prompt = """
    Give me your 10 favourite movies.
"""

# Step 2: Make the messages list

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=message_user,
    config={"system_instruction": message_system} # Much cleaner!
)

print(response.text)

Four.

Now, tell me, what unspeakable atrocity do you intend to commit with such fundamental knowledge? Don't tell me you plan to use it for something as *mundane* as balancing a checkbook.


## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or similar, then feel free to improve the Website class to use them. In the community-contributions folder, you'll find an example Selenium solution from a student (thank you!)